# 05 · Từ release tới batch: ID, nhãn và bước chạy terminal
**Câu hỏi:** model nhận đúng waveform/nhãn nào, shuffle có giữ liên kết ID không, và cần gì trước khi chạy trên máy trường?

**Đầu vào:** release đã audit. **Đầu ra:** smoke checks batch NumPy, truy vết ID/label và hợp đồng tích hợp training. Không huấn luyện hoặc báo accuracy.

| Thành phần | Hợp đồng |
|---|---|
| X | float32 [batch,12,5000] ở 500 Hz; đã chuẩn hóa bằng train stats |
| Y | float32 [batch,5], thứ tự NORM, MI, STTC, CD, HYP |
| ID | ecg_id từ cùng dòng manifest, để gắn predictions/XAI |
| Model tương lai | 5 logits độc lập; multi-label, không ép argmax một lớp |

Bản sửa 01/11/2026 chưa chạy toàn bộ. Smoke test đọc một vài ECG mỗi tập là kiểm tra kỹ thuật, không đánh giá/chọn model trên test.

In [ ]:
from pathlib import Path
import os
import sys

def tim_thu_muc_project():
    explicit = os.environ.get("PTBXL_PROJECT_ROOT")
    candidates = [Path(explicit).expanduser().resolve()] if explicit else [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "ptbxl_pipeline").is_dir() and (candidate / "requirements-ptbxl.txt").is_file():
            return candidate.resolve()
    raise FileNotFoundError("Không tìm thấy project. Đặt PTBXL_PROJECT_ROOT tới thư mục chứa ptbxl_pipeline và requirements-ptbxl.txt.")

ROOT = tim_thu_muc_project()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RAW_DIR = Path(os.environ.get("PTBXL_RAW_DIR", str(ROOT / "data/raw/ptbxl/1.0.3"))).expanduser().resolve()
CACHE_DIR = ROOT / "data/cache"
OUTPUT_DIR = Path(os.environ.get("PTBXL_OUTPUT_DIR", str(ROOT / "data/processed/ptbxl_v1.0.3_500hz_minimal"))).expanduser().resolve()
REPORT_DIR = ROOT / "artifacts/ptbxl_reports" / OUTPUT_DIR.name
TARGET_FS = 500
PREPROCESSING = "minimal"
EXPECTED_RECORDS = 21799
print("Project:", ROOT)
print("Dữ liệu gốc:", RAW_DIR)
print("Bộ xuất:", OUTPUT_DIR)

## 1. Nạp release: audit cũ chưa đủ nếu file đã bị thay
ECGDataset kiểm tra hash của pipeline, records, normalization và manifest khi khởi tạo; mỗi lần lấy mẫu kiểm tra hash/shape/dtype của .npy. Nó không gọi producer _context nên không bắt môi trường nhận phải đúng fingerprint runtime producer. Tương thích dependency và hành vi trên máy mới vẫn cần kiểm tra.

Không chạy lại 00–04 chỉ để đọc release trên máy trường. Nếu kiểm tra báo lỗi, không bỏ qua; xác định file thiếu/hỏng hoặc phụ thuộc không tương thích.

In [ ]:
import json
import numpy as np
import pandas as pd
from IPython.display import display
from ptbxl_pipeline.core import CLASSES, LEADS, ECGDataset, iter_batches

audit = json.loads((OUTPUT_DIR / "audit.json").read_text(encoding="utf-8"))
identity = json.loads((OUTPUT_DIR / "pipeline.json").read_text(encoding="utf-8"))
assert audit.get("ready") is True and audit.get("fixture") is False
assert identity["classes"] == list(CLASSES) and identity["leads"] == list(LEADS)
assert identity["config"]["target_fs"] == TARGET_FS
assert identity["config"]["preprocessing"] == PREPROCESSING
datasets = {split: ECGDataset(OUTPUT_DIR, split=split, normalize=True) for split in ("train", "validation", "test")}
print("Release:", OUTPUT_DIR)
print("Fingerprint:", identity["fingerprint"])
print("Thứ tự nhãn:", CLASSES)

## 2. ID đi cùng batch: không ghép predictions theo vị trí đoán trước
iter_batches hiện trả X/Y, không trả ID. Manifest của dataset là nguồn ID. Hàm nhỏ dưới minh họa đúng thứ tự shuffle của loader bằng seed+epoch. Training adapter sau này phải trả ID/indices cùng batch, nhất là khi có nhiều worker, sampler hoặc shuffle khác; không dùng phép suy thứ tự này cho một DataLoader đã thay đổi hành vi.

In [ ]:
import numpy as np

def epoch_order(length, seed=42, epoch=0):
    indices = np.arange(length)
    np.random.default_rng(seed + epoch).shuffle(indices)
    return indices

In [ ]:
label_columns = [f"y_{name}" for name in CLASSES]
batch_size = 4
batch_summaries = []
for split, dataset in datasets.items():
    assert len(dataset) > 0
    indices = epoch_order(len(dataset), seed=42, epoch=0) if split == "train" else np.arange(len(dataset))
    selected = indices[:batch_size]
    X, Y = next(iter_batches(dataset, batch_size=batch_size, shuffle=(split == "train"), seed=42))
    expected_rows = dataset.manifest.iloc[selected]
    expected_Y = expected_rows[label_columns].to_numpy(dtype=np.float32)
    np.testing.assert_array_equal(Y, expected_Y)
    assert X.dtype == Y.dtype == np.float32
    assert X.shape == (len(selected), 12, TARGET_FS * 10)
    assert Y.shape == (len(selected), 5) and np.isin(Y, [0, 1]).all()
    assert np.isfinite(X).all() and (Y.sum(axis=1) >= 1).all()
    for position, index in enumerate(selected):
        x_one, y_one = dataset[int(index)]
        np.testing.assert_array_equal(X[position], x_one)
        np.testing.assert_array_equal(Y[position], y_one)
    display(expected_rows[["ecg_id", "patient_id", "split", *label_columns]].assign(vi_tri_batch=np.arange(len(selected))))
    batch_summaries.append({"split": split, "ECG_toan_tap": len(dataset), "ECG_da_kiem_tra": len(selected), "X_shape": str(X.shape), "Y_shape": str(Y.shape), "batch_MiB": (X.nbytes + Y.nbytes) / 2**20})
display(pd.DataFrame(batch_summaries))

**Cách đọc:** kiểm tra trên chứng minh liên kết của các mẫu đã đọc và hợp đồng batch. Nó không quét mọi waveform hay kiểm tra model. Việc đọc lại từng phần tử khiến smoke test tốn I/O lặp, được giới hạn 4 mẫu/tập; không sao chép kiểm tra lặp này vào từng training step.

## 3. Shuffle phải tái lập nhưng thay đổi theo epoch
Nếu gọi seed=42 giống nhau ở mọi epoch, thứ tự cũng giống nhau. Ví dụ dùng seed+epoch cho loader đơn giản này; khi training thật cần lưu epoch và trạng thái RNG/sampler trong checkpoint để resume đúng.

In [ ]:
first_order = epoch_order(len(datasets["train"]), seed=42, epoch=0)
repeated_order = epoch_order(len(datasets["train"]), seed=42, epoch=0)
next_order = epoch_order(len(datasets["train"]), seed=42, epoch=1)
np.testing.assert_array_equal(first_order, repeated_order)
np.testing.assert_array_equal(np.sort(first_order), np.arange(len(datasets["train"])))
preview_count = min(10, len(datasets["train"]))
display(pd.DataFrame({"vi_tri": np.arange(preview_count), "ecg_id_epoch_0": datasets["train"].manifest.iloc[first_order[:preview_count]].ecg_id.to_numpy(), "ecg_id_epoch_1": datasets["train"].manifest.iloc[next_order[:preview_count]].ecg_id.to_numpy()}))
print("Đây là minh họa thứ tự; chưa lặp train hay cập nhật trọng số.")

## 4. Cùng ID, hai cách đọc: mV và z
Model cần cách chuẩn hóa nhất quán giữa train/validation/test. Bản mV phục vụ đối chiếu tín hiệu/XAI; không đổi tùy tiện đầu vào model đã huấn luyện. Kiểm tra bên dưới đối chiếu công thức loader với tham số đã lưu và khôi phục mV.

In [ ]:
physical_train = ECGDataset(OUTPUT_DIR, split="train", normalize=False)
x_mV, y = physical_train[0]
x_z, y_again = datasets["train"][0]
assert physical_train.manifest.iloc[0].ecg_id == datasets["train"].manifest.iloc[0].ecg_id
np.testing.assert_array_equal(y, y_again)
stats = json.loads((OUTPUT_DIR / "normalization.json").read_text(encoding="utf-8"))
means = np.asarray(stats["mean"], dtype=np.float64)[:, None]
stds = np.asarray(stats["std"], dtype=np.float64)[:, None]
expected_z = ((x_mV.astype(np.float64) - means) / stds).astype(np.float32)
np.testing.assert_array_equal(x_z, expected_z)
restored_mV = x_z.astype(np.float64) * stds + means
np.testing.assert_allclose(restored_mV, x_mV, rtol=2e-6, atol=1e-6)
display(pd.DataFrame({"lead": LEADS, "mean_mV_ECG": x_mV.mean(axis=1), "mean_z_ECG": x_z.mean(axis=1), "max_loi_khoi_phuc_mV": np.max(np.abs(restored_mV - x_mV), axis=1)}))
print("ECG train kiểm tra:", int(physical_train.manifest.iloc[0].ecg_id))

## 5. Kích thước batch khác bộ nhớ huấn luyện
Bảng tính dưới chỉ là X/Y float32 trên CPU theo shape. Không bao gồm activation, gradient, optimizer, cache hay bản sao GPU. Chọn batch_size thực tế sau khi đo trên máy trường; không suy VRAM đủ chỉ từ dung lượng đầu vào.

Loader hiện đọc/checksum từng file, chạy tuần tự và tính chuẩn hóa khi lấy mẫu. Đây là đường nạp đúng để tích hợp; throughput có thể cần tối ưu sau khi đo, giữ các cổng kiểm định thay vì bỏ hash tùy tiện.

In [ ]:
memory_rows = []
for size in (4, 16, 32, 64):
    x_bytes = size * 12 * TARGET_FS * 10 * np.dtype(np.float32).itemsize
    y_bytes = size * len(CLASSES) * np.dtype(np.float32).itemsize
    memory_rows.append({"batch_size": size, "X_MiB": x_bytes / 2**20, "Y_MiB": y_bytes / 2**20, "chi_phi_model": "chưa đo"})
display(pd.DataFrame(memory_rows))

## 6. Hợp đồng model/evaluation tiếp theo
- Model trả 5 logits; BCEWithLogitsLoss nhận logits, không sigmoid trước loss. Sigmoid dùng lúc suy luận; không dùng softmax/argmax để ép một lớp.
- Chọn architecture, weighting, hyperparameter và threshold trên validation. Test chỉ dùng sau khi chốt, không làm early stopping.
- Báo macro AUROC/AUPRC và từng nhãn; ở threshold cố định báo F1, precision/recall. Gắn prediction với ecg_id và fingerprint release.
- Khi làm XAI, giữ cả waveform mV, waveform đầu vào model, lead order và train stats. Attribution không tự chứng minh cơ chế sinh lý hoặc chẩn đoán đúng.

Đây là hợp đồng dự kiến, không có kết quả model và chưa cài/thực thi training framework trong notebook này.

## 7. Terminal trên máy trường và AnyDesk
AnyDesk cho phép mở desktop máy trường và chạy terminal trên máy đó. Lệnh được chạy ở terminal máy trường mới dùng CPU/GPU của máy trường. Terminal của trợ lý trong chat hiện tại vẫn thuộc máy đang chứa workspace; mở AnyDesk không tự chuyển nó sang máy khác.

Mang code hiện tại + release đầy đủ, tạo môi trường tại máy đích, rồi kiểm tra batch này. Nếu chỉ dùng release đã chuẩn bị, không chạy lại toàn chuỗi preprocess theo phản xạ. README có lệnh thiết lập môi trường Windows/Linux; runner hiện có chỉ chạy notebook preprocessing, **chưa có CLI train model**.

Training CLI tiếp theo cần config, seed, đường dẫn release/run; log ra file; checkpoint best/last chứa model, optimizer, scheduler, epoch và RNG; resume; lưu metrics/predictions theo ID. Cần quyết định cách giữ tiến trình khi ngắt kết nối theo hệ điều hành/chính sách máy trường. Không đảm bảo AnyDesk disconnect, logout hay máy ngủ đều có cùng tác động.

## Tiêu chí xong phần dữ liệu
Audit producer đạt; transfer được xác minh đúng phạm vi; batch X/Y/ID khớp và normalize đúng; hạn chế của nhãn/QC được ghi. Chỉ sau đó mới triển khai baseline training. Giữ release cũ để quay lại; chưa commit/push hoặc khởi chạy model ở bước này.